In [ ]:
# Cell 1: 导入必要的库
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# 设置工作目录
work_dir = r'E:\chuli'
os.chdir(work_dir)

print("第一部分：空间数据处理")
print(f"工作目录: {work_dir}")


In [ ]:
# Cell 2: 加载基础数据
try:
    # 加载学校数据
    schools = pd.read_csv('school_HK.csv')
    print(f"学校数据形状: {schools.shape}")
    print("学校数据列名:", list(schools.columns))
    print(schools.head(2))

    # 加载街景数据
    street_view = pd.read_csv('ratio_HK_individual.csv')
    print(f"\n街景数据形状: {street_view.shape}")
    print("街景数据列名:", list(street_view.columns))
    print(street_view.head(2))

    # 加载交通事故数据
    accidents = pd.read_csv('hongkong_accidents.csv')
    print(f"\n事故数据形状: {accidents.shape}")
    print("事故数据列名:", list(accidents.columns))
    
    if 'Type_of_Co' in accidents.columns:
        print("Type_of_Co唯一值:", list(accidents['Type_of_Co'].unique()))
    if 'Peak_period' in accidents.columns:
        print("Peak_period唯一值:", list(accidents['Peak_period'].unique()))
    print(accidents.head(2))
    
except Exception as e:
    print(f"数据加载错误: {e}")


In [ ]:
# Cell 3: 创建GeoDataFrame和300米缓冲区
try:
    # 创建学校的GeoDataFrame
    geometry_schools = [Point(xy) for xy in zip(schools['longitude'], schools['latitude'])]
    gdf_schools = gpd.GeoDataFrame(schools, geometry=geometry_schools, crs='EPSG:4326')

    # 创建街景数据的GeoDataFrame
    geometry_street = [Point(xy) for xy in zip(street_view['longitude'], street_view['latitude'])]
    gdf_street = gpd.GeoDataFrame(street_view, geometry=geometry_street, crs='EPSG:4326')

    # 创建事故数据的GeoDataFrame
    geometry_accidents = [Point(xy) for xy in zip(accidents['longitude'], accidents['latitude'])]
    gdf_accidents = gpd.GeoDataFrame(accidents, geometry=geometry_accidents, crs='EPSG:4326')

    # 转换坐标系到投影坐标系（米单位），便于计算距离
    gdf_schools_projected = gdf_schools.to_crs('EPSG:3857')  # Web Mercator
    gdf_street_projected = gdf_street.to_crs('EPSG:3857')
    gdf_accidents_projected = gdf_accidents.to_crs('EPSG:3857')

    print("坐标系转换完成")
    print(f"学校数据CRS: {gdf_schools_projected.crs}")
    print(f"街景数据CRS: {gdf_street_projected.crs}")
    print(f"事故数据CRS: {gdf_accidents_projected.crs}")

    # 为每个学校创建300米缓冲区
    gdf_schools_buffer = gdf_schools_projected.copy()
    gdf_schools_buffer['buffer_300m'] = gdf_schools_buffer.geometry.buffer(300)
    gdf_schools_buffer.set_geometry('buffer_300m', inplace=True)

    print("300米缓冲区创建完成")
    print(f"缓冲区数据形状: {gdf_schools_buffer.shape}")
    
except Exception as e:
    print(f"GeoDataFrame创建错误: {e}")


In [ ]:
# Cell 4: 街景数据与学校缓冲区对应
try:
    print("开始街景数据与学校缓冲区匹配...")
    
    # 使用空间连接将街景点与学校缓冲区关联
    street_with_schools = gpd.sjoin(gdf_street_projected, gdf_schools_buffer, 
                                   how='left', predicate='within')

    # 只保留成功匹配的数据
    street_matched = street_with_schools.dropna(subset=['index_right'])
    print(f"成功匹配到学校的街景点数量: {len(street_matched)}")
    
    # 保存街景-学校匹配结果
    cols_to_save = [col for col in street_matched.columns if col != 'geometry']
    streetview_output = os.path.join(work_dir, 'streetview_schools_matched.csv')
    street_matched[cols_to_save].to_csv(streetview_output, index=False)
    print(f"街景-学校匹配结果已保存到 {streetview_output}")
    
except Exception as e:
    print(f"街景数据匹配错误: {e}")


In [ ]:
# Cell 5: 事故数据与学校缓冲区对应
try:
    print("开始事故数据与学校缓冲区匹配...")
    
    # 使用空间连接将事故点与学校缓冲区关联
    accidents_with_schools = gpd.sjoin(gdf_accidents_projected, gdf_schools_buffer, 
                                      how='left', predicate='within')

    # 只保留成功匹配的数据
    accidents_matched = accidents_with_schools.dropna(subset=['index_right'])
    print(f"成功匹配到学校的事故数量: {len(accidents_matched)}")
    
    # 保存事故-学校匹配结果
    cols_to_save = [col for col in accidents_matched.columns if col != 'geometry']
    accidents_output = os.path.join(work_dir, 'accidents_schools_matched.csv')
    accidents_matched[cols_to_save].to_csv(accidents_output, index=False)
    print(f"事故-学校匹配结果已保存到 {accidents_output}")
    
except Exception as e:
    print(f"事故数据匹配错误: {e}")


In [ ]:
# Cell 6: 统计每个学校缓冲区内的街景特征
try:
    print("统计每个学校缓冲区内的街景特征...")
    
    # 明确指定街景特征列
    street_feature_cols = ['wall', 'building', 'sky', 'road', 'tree', 'fence', 'sidewalk', 'pole', 'signboard', 'streetlight']
    
    # 检查这些列是否都存在于数据中
    available_features = [col for col in street_feature_cols if col in street_matched.columns]
    missing_features = [col for col in street_feature_cols if col not in street_matched.columns]
    
    if missing_features:
        print(f"警告: 以下特征列在数据中不存在: {missing_features}")
    
    if available_features:
        print(f"检测到 {len(available_features)} 个街景特征列")
        print("可用的特征列:", available_features)
        
        # 按学校计算街景特征的平均值
        street_stats = street_matched.groupby('School_ID')[available_features].mean().reset_index()
        
        print(f"街景特征统计结果形状: {street_stats.shape}")
        print("街景统计前5行:")
        print(street_stats.head())
        
        # 保存街景特征统计数据
        street_stats_output = os.path.join(work_dir, 'streetview_statistics_by_school.csv')
        street_stats.to_csv(street_stats_output, index=False)
        print(f"街景特征统计数据已保存到 {street_stats_output}")
    else:
        print("错误: 没有可用的街景特征列")
        
except Exception as e:
    print(f"街景特征统计错误: {e}")


In [ ]:
# Cell 7: 统计每个学校缓冲区内的事故情况（按类型和高峰期）
try:
    print("统计每个学校缓冲区内的事故情况...")
    
    # 检查必要字段是否存在
    required_fields = ['School_ID']
    if 'Type_of_Co' in accidents_matched.columns:
        required_fields.append('Type_of_Co')
    if 'Peak_period' in accidents_matched.columns:
        required_fields.append('Peak_period')
    
    available_fields = [field for field in required_fields if field in accidents_matched.columns]
    print(f"可用的分组字段: {available_fields}")
    
    if len(available_fields) >= 2:  # 至少需要School_ID和一个分类字段
        # 按学校、事故类型和高峰期统计
        accident_stats = accidents_matched.groupby(available_fields).size().reset_index(name='accident_count')
        print("事故统计前10行:")
        print(accident_stats.head(10))

        # 创建更清晰的列名
        if len(available_fields) == 3:  # School_ID, Type_of_Co, Peak_period
            # 展开为宽格式便于后续分析
            accident_pivot = accident_stats.pivot_table(
                index='School_ID',
                columns=['Type_of_Co', 'Peak_period'],
                values='accident_count',
                fill_value=0
            ).reset_index()

            # 扁平化多级列名
            if isinstance(accident_pivot.columns, pd.MultiIndex):
                new_columns = []
                for col in accident_pivot.columns:
                    if isinstance(col, tuple):
                        # 将元组转换为字符串，格式为 "Type_Peak"
                        if col[0] == 'School_ID':
                            new_columns.append('School_ID')
                        else:
                            new_columns.append(f"{col[0]}_{col[1]}")
                    else:
                        new_columns.append(str(col))
                accident_pivot.columns = new_columns
            else:
                accident_pivot.columns = ['School_ID'] + [f"accident_{col}" for col in accident_pivot.columns[1:]]

        elif len(available_fields) == 2:  # 只有两个字段的情况
            accident_pivot = accident_stats.pivot_table(
                index='School_ID',
                columns=[col for col in available_fields if col != 'School_ID'][0],
                values='accident_count',
                fill_value=0
            ).reset_index()
            
            # 扁平化列名
            if isinstance(accident_pivot.columns, pd.MultiIndex):
                accident_pivot.columns = ['_'.join([str(x) for x in col]).strip('_') 
                                         if isinstance(col, tuple) 
                                         else col for col in accident_pivot.columns]

        print("\n透视后的事故统计数据前5行:")
        print(accident_pivot.head())

        # 保存事故统计数据
        accident_stats_output = os.path.join(work_dir, 'accident_statistics_by_school.csv')
        accident_pivot.to_csv(accident_stats_output, index=False)
        print(f"事故统计数据已保存到 {accident_stats_output}")
        
    else:
        print(f"缺少必要字段，只进行简单的学校级别统计")
        # 简单统计每个学校的事故总数
        simple_stats = accidents_matched.groupby('School_ID').size().reset_index(name='total_accidents')
        accident_stats_output = os.path.join(work_dir, 'accident_statistics_by_school.csv')
        simple_stats.to_csv(accident_stats_output, index=False)
        print(f"简单事故统计数据已保存到 {accident_stats_output}")
        
except Exception as e:
    print(f"事故统计错误: {e}")


In [ ]:
# Cell 8: 合并所有数据
try:
    print("合并所有数据...")
    
    # 加载统计数据
    street_stats = pd.read_csv('streetview_statistics_by_school.csv')
    
    # 检查事故统计数据是否存在
    if os.path.exists('accident_statistics_by_school.csv'):
        accident_stats = pd.read_csv('accident_statistics_by_school.csv')
        print(f"事故统计数据形状: {accident_stats.shape}")
    else:
        # 创建空的事故统计表
        accident_stats = pd.DataFrame({'School_ID': street_stats['School_ID']})
        print("未找到事故统计数据，创建空表")
    
    print(f"街景统计数据形状: {street_stats.shape}")
    
    # 合并数据
    final_merged_data = pd.merge(street_stats, accident_stats, on='School_ID', how='outer')
    
    print(f"最终合并数据形状: {final_merged_data.shape}")
    print("合并后数据前5行:")
    print(final_merged_data.head())
    
    # 保存最终合并数据
    final_output = os.path.join(work_dir, 'final_analysis_data.csv')
    final_merged_data.to_csv(final_output, index=False)
    print(f"最终分析数据已保存到 {final_output}")
    
    print("\n数据处理完成！所有中间文件已生成。")
    print("生成的文件:")
    for file in ['streetview_schools_matched.csv', 'accidents_schools_matched.csv', 
                 'streetview_statistics_by_school.csv', 'accident_statistics_by_school.csv', 
                 'final_analysis_data.csv']:
        if os.path.exists(file):
            print(f"  ✓ {file}")
    
except Exception as e:
    print(f"数据合并错误: {e}")


In [ ]:
# Cell 1: 导入必要的库
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# 设置工作目录
work_dir = r'E:\chuli'
os.chdir(work_dir)

print("第二部分：随机森林建模分析")
print(f"工作目录: {work_dir}")


In [ ]:
# Cell 2: 加载并检查数据
try:
    # 加载最终合并的数据
    merged_data = pd.read_csv('final_analysis_data.csv')
    
    print("数据加载完成")
    print(f"数据形状: {merged_data.shape}")
    print(f"总列数: {len(merged_data.columns)}")
    
    # 显示基本信息
    print(f"\n数据基本信息:")
    print(f"- 学校数量: {merged_data['School_ID'].nunique()}")
    print(f"- 数值列数量: {len(merged_data.select_dtypes(include=[np.number]).columns)}")
    print(f"- 非数值列数量: {len(merged_data.select_dtypes(exclude=[np.number]).columns)}")
    
    # 显示前几行
    print(f"\n数据前5行:")
    print(merged_data.head())
    
    # 显示列名
    print(f"\n所有列名:")
    column_list = list(merged_data.columns)
    for i, col in enumerate(column_list):
        print(f"  {i+1:2d}. {col}")
    
except Exception as e:
    print(f"数据加载错误: {e}")


In [ ]:
# Cell 3: 数据预处理和目标变量识别
def identify_accident_columns(data):
    """
    识别事故相关的统计列作为目标变量
    """
    # 查找包含事故统计的列
    accident_cols = [col for col in data.columns if col.startswith(('Collision', 'Pedestrian', 'Vehicle', 'accident')) 
                     or 'accident' in col.lower()]
    
    # 排除学校ID等非目标变量
    target_variables = [col for col in accident_cols if col != 'School_ID' and 'total' not in col.lower()]
    
    return target_variables

def prepare_modeling_data(data):
    """
    准备建模用的数据
    """
    # 识别事故目标变量
    accident_cols = identify_accident_columns(data)
    
    # 特征列（排除School_ID和事故统计列）
    feature_columns = [col for col in data.columns if col not in accident_cols and col != 'School_ID']
    
    # 处理缺失值
    X = data[feature_columns].fillna(data[feature_columns].mean())
    y_dict = {}
    
    # 为每个目标变量准备数据
    for target_col in accident_cols:
        if target_col in data.columns:
            y = data[target_col].fillna(0)  # 事故数量用0填充
            y_dict[target_col] = y
    
    return X, y_dict, feature_columns

# 识别目标变量
target_variables = identify_accident_columns(merged_data)
print("识别到的事故目标变量:")
for i, var in enumerate(target_variables, 1):
    sample_value = merged_data[var].iloc[0] if var in merged_data.columns else "N/A"
    print(f"{i:2d}. {var} (示例值: {sample_value})")

# 准备建模数据
X, y_dict, feature_names = prepare_modeling_data(merged_data)
print(f"\n特征数量: {len(feature_names)}")
print(f"目标变量数量: {len(y_dict)}")
print("前10个特征:", feature_names[:10])


In [ ]:
# Cell 4: 随机森林模型训练和评估函数
def train_regression_model(X, y, target_name, test_size=0.2, random_state=42):
    """
    训练随机森林回归模型
    """
    print(f"  开始训练回归模型: {target_name}")
    try:
        # 检查数据
        print(f"  数据检查 - X形状: {X.shape}, y形状: {y.shape}")
        print(f"  y的统计信息: 均值={y.mean():.3f}, 标准差={y.std():.3f}, 最小值={y.min()}, 最大值={y.max()}")
        
        # 分割数据
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
        print(f"  数据分割完成 - 训练集: {X_train.shape}, 测试集: {X_test.shape}")
        
        # 标准化特征
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        print(f"  数据标准化完成")
        
        # 训练随机森林回归模型
        print(f"  开始训练随机森林模型...")
        rf_model = RandomForestRegressor(n_estimators=100, random_state=random_state, n_jobs=-1)
        rf_model.fit(X_train_scaled, y_train)
        print(f"  模型训练完成")
        
        # 预测
        y_pred = rf_model.predict(X_test_scaled)
        print(f"  预测完成")
        
        # 评估
        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        print(f"  评估完成 - MSE: {mse:.4f}, R2: {r2:.4f}")
        
        # 特征重要性
        feature_importance = pd.DataFrame({
            'feature': X.columns,
            'importance': rf_model.feature_importances_
        }).sort_values('importance', ascending=False)
        print(f"  特征重要性计算完成")
        
        result = {
            'model': rf_model,
            'scaler': scaler,
            'mse': mse,
            'r2': r2,
            'y_test': y_test,
            'y_pred': y_pred,
            'feature_importance': feature_importance,
            'target_name': target_name,
            'success': True
        }
        print(f"  回归模型 {target_name} 训练成功")
        return result
        
    except Exception as e:
        print(f"  回归模型训练错误 {target_name}: {e}")
        import traceback
        traceback.print_exc()
        return {'success': False, 'error': str(e)}

def train_classification_model(X, y, target_name, test_size=0.2, random_state=42):
    """
    训练随机森林分类模型
    """
    print(f"  开始训练分类模型: {target_name}")
    try:
        # 将连续值转换为二分类（是否有事故）
        y_binary = (y > 0).astype(int)
        
        # 检查类别分布
        unique_classes, counts = np.unique(y_binary, return_counts=True)
        print(f"  类别分布: {dict(zip(unique_classes, counts))}")
        
        if len(unique_classes) < 2:
            print(f"  警告: 目标变量 {target_name} 只有一个类别，无法进行分类")
            return {'success': False, 'error': 'Only one class'}
        
        # 分割数据
        X_train, X_test, y_train, y_test = train_test_split(X, y_binary, test_size=test_size, 
                                                            random_state=random_state, 
                                                            stratify=y_binary)
        print(f"  数据分割完成 - 训练集: {X_train.shape}, 测试集: {X_test.shape}")
        
        # 标准化特征
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        print(f"  数据标准化完成")
        
        # 训练随机森林分类器
        print(f"  开始训练随机森林分类器...")
        rf_classifier = RandomForestClassifier(n_estimators=100, random_state=random_state, n_jobs=-1)
        rf_classifier.fit(X_train_scaled, y_train)
        print(f"  分类器训练完成")
        
        # 预测
        y_pred = rf_classifier.predict(X_test_scaled)
        print(f"  预测完成")
        
        # 评估
        accuracy = accuracy_score(y_test, y_pred)
        print(f"  准确率计算完成: {accuracy:.4f}")
        
        # 特征重要性
        feature_importance = pd.DataFrame({
            'feature': X.columns,
            'importance': rf_classifier.feature_importances_
        }).sort_values('importance', ascending=False)
        print(f"  特征重要性计算完成")
        
        result = {
            'model': rf_classifier,
            'scaler': scaler,
            'accuracy': accuracy,
            'y_test': y_test,
            'y_pred': y_pred,
            'classification_report': classification_report(y_test, y_pred, output_dict=True),
            'feature_importance': feature_importance,
            'target_name': target_name,
            'success': True
        }
        print(f"  分类模型 {target_name} 训练成功")
        return result
        
    except Exception as e:
        print(f"  分类模型训练错误 {target_name}: {e}")
        import traceback
        traceback.print_exc()
        return {'success': False, 'error': str(e)}

def plot_feature_importance(feature_importance, target_name, top_n=15, save_plot=True):
    """
    绘制特征重要性图
    """
    print(f"  开始绘制特征重要性图: {target_name}")
    try:
        plt.figure(figsize=(10, 8))
        top_features = feature_importance.head(top_n)
        bars = plt.barh(range(len(top_features)), top_features['importance'])
        plt.yticks(range(len(top_features)), top_features['feature'], fontsize=8)
        plt.xlabel('Feature Importance')
        plt.title(f'Top {top_n} Feature Importance for {target_name}')
        plt.gca().invert_yaxis()
        plt.tight_layout()
        
        if save_plot:
            filename = f'feature_importance_{target_name.replace("/", "_").replace(" ", "_")}.png'
            filepath = os.path.join(work_dir, filename)
            plt.savefig(filepath, dpi=300, bbox_inches='tight')
            print(f"  特征重要性图已保存到: {filename}")
        
        plt.show()
        plt.close()
        print(f"  特征重要性图绘制完成")
    except Exception as e:
        print(f"  绘图错误: {e}")
        import traceback
        traceback.print_exc()
        plt.close()


In [ ]:
# Cell 5: 按事故类型和高峰期进行回归分析（修正版）
print("=== 回归分析：预测各类事故数量 ===")

regression_results = {}
successful_reg_models = 0

print(f"目标变量字典大小: {len(y_dict)}")
print(f"目标变量列表: {list(y_dict.keys())}")

# 对每个目标变量进行回归分析
for target_var, y_values in y_dict.items():
    print(f"\n{'='*60}")
    print(f"开始分析目标变量: {target_var}")
    print(f"y_values 类型: {type(y_values)}, 形状: {y_values.shape if hasattr(y_values, 'shape') else '无形状属性'}")
    print(f"y_values 前5个值: {y_values.head() if hasattr(y_values, 'head') else y_values[:5]}")
    
    # 检查数据质量
    if len(y_values) > 10 and y_values.var() > 0:
        print(f"数据质量检查通过，开始训练模型...")
        # 训练回归模型
        result = train_regression_model(X, y_values, target_var)
        
        if result['success']:
            regression_results[target_var] = result
            successful_reg_models += 1
            print(f"✓ 模型 {target_var} 训练成功")
        else:
            print(f"✗ 模型 {target_var} 训练失败: {result.get('error', 'Unknown error')}")
    else:
        print(f"✗ 数据质量不足: 样本数={len(y_values)}, 方差={y_values.var():.6f}")

print(f"\n{'='*60}")
print(f"回归分析完成！成功训练 {successful_reg_models} 个模型")


In [ ]:
# Cell 6: 按事故类型和高峰期进行分类分析
print("\n=== 分类分析：预测是否发生各类事故 ===")

classification_results = {}
successful_class_models = 0

# 对每个目标变量进行分类分析
for target_var, y_values in y_dict.items():
    print(f"\n{'='*60}")
    print(f"分类分析目标: {target_var}")
    print(f"样本数量: {len(y_values)}, 正样本比例: {(y_values > 0).mean():.2%}")
    
    # 检查类别分布
    positive_samples = (y_values > 0).sum()
    negative_samples = (y_values == 0).sum()
    print(f"正样本(有事故): {positive_samples}, 负样本(无事故): {negative_samples}")
    
    if positive_samples > 5 and negative_samples > 5:  # 确保有足够的两类样本
        # 训练分类模型
        result = train_classification_model(X, y_values, target_var)
        
        if result['success']:
            classification_results[target_var] = result
            successful_class_models += 1
            
            print(f"✓ 分类模型训练成功")
            print(f"  Accuracy: {result['accuracy']:.4f}")
            
            # 显示分类报告摘要
            if 'classification_report' in result:
                report = result['classification_report']
                if isinstance(report, dict):
                    print("  分类报告摘要:")
                    for class_label, metrics in report.items():
                        if isinstance(metrics, dict) and 'precision' in metrics and str(class_label).isdigit():
                            print(f"    类别 {class_label}: Precision={metrics['precision']:.3f}, "
                                  f"Recall={metrics['recall']:.3f}, F1={metrics['f1-score']:.3f}")
            
            # 显示最重要的特征
            print("  Top 10 Important Features:")
            top_features = result['feature_importance'].head(10)
            for idx, (_, row) in enumerate(top_features.iterrows(), 1):
                print(f"    {idx:2d}. {row['feature']}: {row['importance']:.4f}")
            
            # 绘制特征重要性
            plot_feature_importance(result['feature_importance'], f"{target_var}_classification", top_n=10)
        else:
            print(f"✗ 分类模型训练失败: {result.get('error', 'Unknown error')}")
    else:
        print(f"✗ 样本分布不均衡: 正样本={positive_samples}, 负样本={negative_samples}")

print(f"\n{'='*60}")
print(f"分类分析完成！成功训练 {successful_class_models} 个模型")


In [ ]:
# Cell 7: 相关性分析和特征重要性汇总
print("\n=== 相关性分析和特征重要性汇总 ===")

# 全局特征重要性分析
print("\n1. 全局特征重要性分析:")
all_feature_importances = []

# 收集所有模型的特征重要性
for target_var, result in regression_results.items():
    if result['success']:
        fi = result['feature_importance'].copy()
        fi['target_variable'] = target_var
        fi['model_type'] = 'regression'
        all_feature_importances.append(fi[['feature', 'importance', 'target_variable', 'model_type']])

for target_var, result in classification_results.items():
    if result['success']:
        fi = result['feature_importance'].copy()
        fi['target_variable'] = target_var
        fi['model_type'] = 'classification'
        all_feature_importances.append(fi[['feature', 'importance', 'target_variable', 'model_type']])

if all_feature_importances:
    combined_fi = pd.concat(all_feature_importances, ignore_index=True)
    # 计算每个特征在所有模型中的平均重要性
    avg_importance = combined_fi.groupby('feature')['importance'].agg(['mean', 'std', 'count']).sort_values('mean', ascending=False)
    avg_importance.columns = ['avg_importance', 'std_importance', 'model_count']
    
    print("   Top 20 most important street view features across all models:")
    for i, (feature, row) in enumerate(avg_importance.head(20).iterrows(), 1):
        print(f"     {i:2d}. {feature}: 平均重要性={row['avg_importance']:.4f}, "
              f"标准差={row['std_importance']:.4f}, 模型数={int(row['model_count'])}")
    
    # 保存特征重要性汇总
    feature_summary_output = os.path.join(work_dir, 'feature_importance_summary.csv')
    avg_importance.to_csv(feature_summary_output)
    print(f"\n✓ 特征重要性汇总已保存到 {feature_summary_output}")
    
else:
    print("   没有特征重要性数据可汇总")

# 相关性热力图（如果有足够数据）
try:
    if len(merged_data.columns) > 2:
        # 选择数值列进行相关性分析
        numeric_data = merged_data.select_dtypes(include=[np.number])
        if len(numeric_data.columns) > 2:
            # 计算相关性矩阵
            corr_matrix = numeric_data.corr()
            
            # 选择部分列绘制热力图（避免过于复杂）
            selected_cols = numeric_data.columns[:min(20, len(numeric_data.columns))]  # 最多20列
            selected_corr = corr_matrix.loc[selected_cols, selected_cols]
            
            # 绘制相关性热力图
            plt.figure(figsize=(12, 10))
            sns.heatmap(selected_corr, annot=False, cmap='coolwarm', center=0, 
                        square=True, linewidths=0.5)
            plt.title('Correlation Matrix of Features and Accidents (Top 20)')
            plt.tight_layout()
            
            corr_plot_path = os.path.join(work_dir, 'correlation_heatmap.png')
            plt.savefig(corr_plot_path, dpi=300, bbox_inches='tight')
            print(f"✓ 相关性热力图已保存到 {corr_plot_path}")
            plt.show()
        else:
            print("数值列不足，跳过相关性热力图")
    else:
        print("列数不足，跳过相关性分析")
        
except Exception as e:
    print(f"相关性分析错误: {e}")
    plt.close()


In [ ]:
# Cell 8: 结果总结和报告
print("\n=== 分析结果总结 ===")

# 回归模型结果总结
print("\n1. 回归模型性能:")
regression_summary = []
if regression_results:
    sorted_results = sorted(regression_results.items(), key=lambda x: x[1]['r2'] if x[1]['success'] else -1, reverse=True)
    for target_var, result in sorted_results:
        if result['success']:
            print(f"   {target_var}:")
            print(f"     R² = {result['r2']:.3f}, MSE = {result['mse']:.3f}")
            regression_summary.append({
                'target_variable': target_var,
                'model_type': 'regression',
                'r2_score': result['r2'],
                'mse': result['mse'],
                'sample_size': len(result['y_test']),
                'top_features': ', '.join(result['feature_importance'].head(3)['feature'].tolist())
            })
else:
    print("   没有成功的回归模型")

# 分类模型结果总结
print("\n2. 分类模型性能:")
classification_summary = []
if classification_results:
    sorted_results = sorted(classification_results.items(), key=lambda x: x[1]['accuracy'] if x[1]['success'] else -1, reverse=True)
    for target_var, result in sorted_results:
        if result['success']:
            print(f"   {target_var}:")
            print(f"     Accuracy = {result['accuracy']:.3f}")
            classification_summary.append({
                'target_variable': target_var,
                'model_type': 'classification',
                'accuracy': result['accuracy'],
                'sample_size': len(result['y_test']),
                'top_features': ', '.join(result['feature_importance'].head(3)['feature'].tolist())
            })
else:
    print("   没有成功的分类模型")

# 生成详细报告
print("\n3. 生成详细分析报告...")

try:
    # 创建结果保存目录
    results_dir = os.path.join(work_dir, 'analysis_results')
    if not os.path.exists(results_dir):
        os.makedirs(results_dir)
    
    # 保存回归分析结果
    if regression_summary:
        regression_df = pd.DataFrame(regression_summary)
        regression_output = os.path.join(results_dir, 'regression_analysis_results.csv')
        regression_df.to_csv(regression_output, index=False)
        print(f"✓ 回归分析结果已保存到 {regression_output}")
    else:
        print("- 没有回归分析结果可保存")

    # 保存分类分析结果
    if classification_summary:
        classification_df = pd.DataFrame(classification_summary)
        classification_output = os.path.join(results_dir, 'classification_analysis_results.csv')
        classification_df.to_csv(classification_output, index=False)
        print(f"✓ 分类分析结果已保存到 {classification_output}")
    else:
        print("- 没有分类分析结果可保存")

    # 保存所有模型的详细结果
    all_model_details = []
    for target_var, result in regression_results.items():
        if result['success']:
            all_model_details.append({
                'target_variable': target_var,
                'model_type': 'regression',
                'metric_1': f"R2:{result['r2']:.3f}",
                'metric_2': f"MSE:{result['mse']:.3f}",
                'sample_size': len(result['y_test'])
            })
    
    for target_var, result in classification_results.items():
        if result['success']:
            all_model_details.append({
                'target_variable': target_var,
                'model_type': 'classification',
                'metric_1': f"Accuracy:{result['accuracy']:.3f}",
                'metric_2': '',
                'sample_size': len(result['y_test'])
            })
    
    if all_model_details:
        model_details_df = pd.DataFrame(all_model_details)
        model_details_output = os.path.join(results_dir, 'all_model_summary.csv')
        model_details_df.to_csv(model_details_output, index=False)
        print(f"✓ 模型汇总已保存到 {model_details_output}")

    # 保存总结报告
    summary_output = os.path.join(results_dir, 'comprehensive_analysis_report.txt')
    with open(summary_output, 'w', encoding='utf-8') as f:
        f.write("交通事故与街景元素相关性分析综合报告\n")
        f.write("=" * 60 + "\n\n")
        
        f.write("一、数据概况\n")
        f.write(f"总学校数量: {merged_data['School_ID'].nunique()}\n")
        f.write(f"总特征数量: {len(feature_names)}\n")
        f.write(f"事故统计变量数量: {len(target_variables)}\n\n")
        
        f.write("二、回归模型性能 (按R²排序)\n")
        if regression_summary:
            for item in sorted(regression_summary, key=lambda x: x['r2_score'], reverse=True):
                f.write(f"  {item['target_variable']}:\n")
                f.write(f"    R² = {item['r2_score']:.3f}, MSE = {item['mse']:.3f}\n")
                f.write(f"    关键特征: {item['top_features']}\n\n")
        else:
            f.write("  无成功模型\n\n")
        
        f.write("三、分类模型性能 (按准确率排序)\n")
        if classification_summary:
            for item in sorted(classification_summary, key=lambda x: x['accuracy'], reverse=True):
                f.write(f"  {item['target_variable']}:\n")
                f.write(f"    Accuracy = {item['accuracy']:.3f}\n")
                f.write(f"    关键特征: {item['top_features']}\n\n")
        else:
            f.write("  无成功模型\n\n")
        
        f.write("四、全局特征重要性 (Top 20)\n")
        if all_feature_importances:
            for i, (feature, row) in enumerate(avg_importance.head(20).iterrows(), 1):
                f.write(f"  {i:2d}. {feature}: {row['avg_importance']:.4f}\n")
        
        f.write(f"\n五、分析时间: {pd.Timestamp.now()}\n")
        f.write(f"六、工作目录: {work_dir}\n")

    print(f"✓ 综合分析报告已保存到 {summary_output}")
    print(f"\n🎉 分析完成！所有结果文件已保存到 '{results_dir}' 目录。")
    
except Exception as e:
    print(f"❌ 结果保存错误: {e}")

print(f"\n工作目录文件清单:")
for file in os.listdir(work_dir):
    if file.endswith(('.csv', '.png', '.txt')) or 'analysis_results' in file:
        file_path = os.path.join(work_dir, file)
        if os.path.isfile(file_path):
            size_mb = os.path.getsize(file_path) / (1024 * 1024)
            print(f"  {file} ({size_mb:.2f} MB)")
        elif os.path.isdir(file_path):
            print(f"  {file}/")
